# Step 1 : Read File 

In [9]:
import pypdf

def reader(file):
    pdf = pypdf.PdfReader(file)
    text = ""

    num_of_pages = len(pdf.pages)

    for page_number in range(num_of_pages):
        current_page = pdf.pages[page_number]
        extracted_text = current_page.extract_text()
        if extracted_text:
            text += extracted_text + "\n"

    print("\nThese are the first 500 characters:\n", text[:500])
    print("\nThis is the number of characters in the paper:\n", len(text))
    print("\nThis is the number of words in the paper:\n", len(text.split()))

    return text

# Step 2: Initializing Model

In [10]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

# Se cargan variables generales

api_key = os.getenv("API_KEY")
model_id = os.getenv("MODELO_ID") # Dato curioso, el model ID, no es un ID, es un nombre 

# Initialize Software Development Kit
client = genai.Client(api_key=api_key)

def get_response_gemini(contents, system_instruction=None, stream=False):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0,
        max_output_tokens=7777
    )

    method = (
        client.models.generate_content_stream
        if stream
        else client.models.generate_content
    )

    return method(
        model=model_id,
        contents=contents,
        config=config
    )


# Step 3: Chat with context

## 3.1 'build_system_prompt' function.

Hever, here we organize a prompt (as you do it with claude) Hahahahaha. But talking seriouly, this function, help us, for the construction of a prompt that includes rol, tone for answering, and ofcourse the context for answering. If questions ask me. 

In [26]:
def build_system_prompt(document_text):
    return f"""
        You are an expert assistant on the paper "Attention Is All You Need".

        You must answer only with information explicitly contained in the document below.
        Do not ask the user to provide the paper, because the full document is already included in this prompt.
        Do not use outside knowledge.
        If the answer is not explicitly in the document, say:
        "The answer is not explicitly available in the provided document."

        Answer in a direct, technical, and concise way.

        Document:
        --------------------------------------------------------------------
        {document_text}
        --------------------------------------------------------------------
        """

# 3.1.2 Normalize content

In [38]:
def normalize_content(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        texts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                texts.append(item.get("text", ""))
        return "\n".join(texts)

    if isinstance(content, dict):
        if content.get("type") == "text":
            return content.get("text", "")
        return str(content)

    return str(content)

## 3.2 'Chat_with_document' function

This is a import function, because it calls the order two functions. In this ideas order, we need to talk with professor, because I modified the initializing client chunck, so we can reuse it on chat_with_document function. In my opinion is cleaner. 

In [39]:
from google.genai import types

def chat_with_document(message, history, text):
    system_instructions = build_system_prompt(text)

    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        clean_content = normalize_content(msg["content"])

        if clean_content.strip():
            gemini_history.append(
                types.Content(
                    role=role,
                    parts=[types.Part(text=clean_content)]
                )
            )

    clean_message = normalize_content(message)

    current_message = types.Content(
        role="user",
        parts=[types.Part(text=clean_message)]
    )

    response_stream = get_response_gemini(
        contents=gemini_history + [current_message],
        system_instruction=system_instructions,
        stream=True
    )

    partial = ""
    for chunk in response_stream:
        if getattr(chunk, "text", None):
            partial += chunk.text
            yield partial

# 4. Gradio Interface

In [45]:
import gradio as gr

document_text = reader("attention_is_all_you_need.pdf")

demo = gr.ChatInterface(
    fn=chat_with_document,
    title="Experto en 'Attention Is All You Need'",
    description="Asistente técnico basado únicamente en el paper de Transformers. Este señor solo le va a responder si las preguntas tienen respuesta en el texto, de lo contrario no.",
    additional_inputs=[
        gr.Textbox(value=document_text, visible=False)
    ],
    examples=[
        ["¿Qué es el mecanismo de Multi-Head Attention?", document_text],
        ["¿Cuáles fueron los resultados en tareas de traducción?", document_text],
        ["Explica el concepto de Positional Encoding.", document_text]
    ]
)

if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=8083,
        debug=True,
        show_error=True
    )


These are the first 500 characters:
 Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 

This is the number of characters in the paper:
 39630

This is the number of words in the paper:
 6141
* Running on local URL:  http://0.0.0.0:8083
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
